In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Toxic Comment Classification using NLP and N-gram Vectorization

## BUSINESS OVERVIEW

Moderating user-generated content is essential for maintaining healthy online communities. Toxic language such as insults, threats, hate speech, and abusive content can damage user experience and platform safety.

The objective of this project is to build a Natural Language Processing model that can automatically identify toxic content from comment text. This can help improve moderation workflows and reduce the workload of human moderators.



## DATASET INFORMATION

The main goal of this project is to predict which toxicity categories apply to a given comment based on its text.

The model generates predictions for the following labels:

- toxic
- severe_toxic
- obscene
- threat
- insult
- identity_hate

Therefore, this is not a single-target classification problem, but a multi-label classification problem.

## PROJECT TARGET

There is no single target variable in this project. The model predicts 6 different toxicity labels at the same time.

The target labels are:

- toxic
- severe_toxic
- obscene
- threat
- insult
- identity_hate

This makes the task a multi-label classification problem.

## LOADING LIBRARIES

In [ ]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

import re
import string

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

## LOAD DATASET

In [ ]:
train = pd.read_csv("/kaggle/input/competitions/jigsaw-toxic-comment-classification-challenge/train.csv.zip")
test = pd.read_csv("/kaggle/input/competitions/jigsaw-toxic-comment-classification-challenge/test.csv.zip")
sample_sub = pd.read_csv("/kaggle/input/competitions/jigsaw-toxic-comment-classification-challenge/sample_submission.csv.zip")
test_labels = pd.read_csv("/kaggle/input/competitions/jigsaw-toxic-comment-classification-challenge/test_labels.csv.zip")

train.shape, test.shape

## EDA

In [ ]:
train.head()    

In [ ]:
train.info()

#### LABEL DISTRIBUTION

In [ ]:
labels = train.columns[2:]

train[labels].sum().sort_values(ascending=False).plot(kind="bar", figsize=(10,5))

plt.title("Toxic Comment Label Distribution")
plt.ylabel("Number of Comments")
plt.xlabel("Toxic Categories")

plt.show()

#### COMMENT LENGTH ANALYSIS

We analyze the length of comments to understand the distribution of text sizes in the dataset.

In [ ]:
train["comment_length"] = train["comment_text"].str.len()

In [ ]:
plt.figure(figsize=(10,5))

sns.histplot(train["comment_length"], bins=50)

plt.title("Comment Length Distribution")

plt.show()

The distribution of comment lengths shows that most comments are relatively short.
The majority of comments fall between 0 and 500 characters, while a smaller number of comments are significantly longer.

The distribution is right-skewed, meaning there are a few very long comments extending beyond 2000 characters.

This observation is important for Natural Language Processing because:

Most comments contain limited text information.

Extremely long comments are rare.


#### Toxic vs Non-Toxic Comment Length

In [ ]:
plt.figure(figsize=(10,5))

sns.boxplot(x=train["toxic"], y=train["comment_length"])

plt.title("Comment Length vs Toxicity")

plt.show()

The boxplot compares the length of toxic and non-toxic comments.

From the visualization we observe:

The median length of toxic comments is slightly shorter than non-toxic comments.

Both categories contain extreme outliers, with some comments exceeding 4000–5000 characters.

The majority of comments for both classes are concentrated below 500 characters.

This indicates that comment length alone is not a strong indicator of toxicity.
Instead, the content and wording of the comment play a more important role.


#### Toxic Label Correlation Heatmap

In [ ]:
plt.figure(figsize=(8,6))

sns.heatmap(train.iloc[:,2:8].corr(), annot=True, cmap="coolwarm")

plt.title("Toxic Label Correlation")

plt.show()

## NLP Feature Engineering

1️⃣ lowercase
2️⃣ punctuation removal
3️⃣ number removal
4️⃣ stopwords removal
5️⃣ lemmatization
6️⃣ vectorization

In [ ]:
train["comment_text"] = train["comment_text"].str.lower()         # 1️⃣ lowercase   
train["comment_text"]=train["comment_text"].str.replace(r"[^\w\s]", "", regex=True) # 2️⃣ punctuation removal 
train["comment_text"]=train["comment_text"].str.replace(r"\d+", "", regex=True) #3️⃣ Number Removal
train["comment_text"]=train["comment_text"].str.replace("\n", " ") #Remove Newline Characters
train["comment_text"]=train["comment_text"].str.replace("\r"," " ,regex=True) #Remove line braks

In [ ]:
import nltk
from nltk.corpus import stopwords
nltk.download('stopwords')
stop_words = set(stopwords.words('english')) # 4️⃣ stopwords removal 

In [ ]:
from nltk.tokenize import word_tokenize


train["comment_text"] = train["comment_text"].apply(lambda x: [word for word in word_tokenize(x) if word not in stop_words])


train["comment_text"].head() # Tokenization

In [ ]:
from nltk.stem import WordNetLemmatizer
nltk.download('wordnet')
nltk.download('omw-1.4')
lemmatizer = WordNetLemmatizer()

train["comment_text"] = train["comment_text"].apply(lambda x: [lemmatizer.lemmatize(word) for word in x]) # 5️⃣ lemmatization

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer

In [ ]:
train["comment_text"] = train["comment_text"].apply(lambda x: " ".join(x) if isinstance(x, list) else x) # 6️⃣ vectorization

## Modelling

In [ ]:
x = train['comment_text']
y = train[['toxic', 'severe_toxic', 'obscene', 'threat', 'insult', 'identity_hate']]

In [ ]:
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=42)

In [ ]:
vect = CountVectorizer(ngram_range=(1,2))

In [ ]:
x_train_vec=vect.fit_transform(x_train)
x_test_vec=vect.transform(x_test)

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
vect = TfidfVectorizer(ngram_range=(1,2))
x_sayisal = vect.fit_transform(train['comment_text']) 
y = train[['toxic', 'severe_toxic', 'obscene', 'threat', 'insult', 'identity_hate']]

In [ ]:
from sklearn.multiclass import OneVsRestClassifier
from sklearn.linear_model import LogisticRegression

In [ ]:
model = OneVsRestClassifier(LogisticRegression(max_iter=1000))

In [ ]:
from sklearn.model_selection import train_test_split

x_train, x_test, y_train, y_test = train_test_split(
    x_sayisal, y, test_size=0.2, random_state=42
)

In [ ]:
model.fit(x_train, y_train)

In [ ]:
pred = model.predict(x_test)

In [ ]:
from sklearn.metrics import f1_score

f1_score(y_test, pred, average="micro")

In [ ]:
from sklearn.metrics import f1_score, precision_score, recall_score

label_scores = pd.DataFrame({
    "Label": y.columns,
    "Precision": precision_score(y_test, pred, average=None, zero_division=0),
    "Recall": recall_score(y_test, pred, average=None, zero_division=0),
    "F1 Score": f1_score(y_test, pred, average=None, zero_division=0)
})

label_scores.sort_values("F1 Score", ascending=False)

In [ ]:
plt.figure(figsize=(8,4))

sns.barplot(data=label_scores.sort_values("F1 Score", ascending=False), x="Label", y="F1 Score")

plt.title("F1 Score by Toxicity Label")
plt.xlabel("Label")
plt.ylabel("F1 Score")
plt.ylim(0, 1)

plt.show()

# CONCLUSION

In this project, a Natural Language Processing pipeline was developed to classify toxic comments from Wikipedia discussions. Text data was transformed into numerical representations using n-gram based vectorization, allowing the model to capture both individual words and short word combinations.

The evaluation results reveal that the model performs relatively well on more frequent toxicity categories such as toxic and obscene, achieving F1 scores around 0.63. The model also shows moderate performance for the insult category with an F1 score of approximately 0.50. These results indicate that the model can successfully learn common toxic language patterns that frequently appear in the dataset.

However, the model struggles with less frequent labels such as severe_toxic, identity_hate, and threat. In particular, the threat category shows a very low F1 score (around 0.08), while identity_hate remains difficult to detect due to its extremely low recall. This is primarily caused by class imbalance, as these categories appear much less frequently in the dataset.

The precision scores across most labels are relatively high, suggesting that when the model predicts a toxic category, it is often correct. However, the recall values for rare classes remain low, meaning the model fails to detect many true toxic cases in those categories.

Overall, the model demonstrates that n-gram based text vectorization combined with classical machine learning models can effectively detect common toxic patterns in online comments. Nevertheless, performance could be further improved by addressing class imbalance and by adopting more advanced Natural Language Processing approaches such as TF-IDF weighting, class balancing techniques, or transformer-based language models.

In [ ]:
# Submission

test_vec = vect.transform(test["comment_text"])

pred_test = model.predict(test_vec)

submission = pd.DataFrame(pred_test, columns=y.columns)

submission.insert(0,"id",test["id"])

submission.to_csv("submission.csv", index=False)

In [ ]:
import joblib
from sklearn.multiclass import OneVsRestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.feature_extraction.text import TfidfVectorizer

x= train["comment_text"]
y = train[["toxic", "severe_toxic", "obscene", "threat", "insult", "identity_hate"]]

vectorizer = TfidfVectorizer(ngram_range=(1, 2), max_features=20000)
x_vec = vectorizer.fit_transform(x)

model = OneVsRestClassifier(LogisticRegression(max_iter=1000))
model.fit(x_vec, y)

joblib.dump(model, "toxic_model.pkl")
joblib.dump(vectorizer, "toxic_vectorizer.pkl")
joblib.dump(y.columns.tolist(), "toxic_columns.pkl")